<a href="https://colab.research.google.com/github/coweye1/CCMEO-Gunshot-Wound-Entrance-vs-Exit-CNN-ViT-Hybrid-Benchmark/blob/main/CCMEO_Gunshot_Wound_Entrance_vs_Exit_CLIP_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 0: Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# 수정 반영된 Step 1 코드
!pip install -q timm scikit-learn pandas matplotlib tqdm openpyxl transformers git+https://github.com/openai/CLIP.git

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import clip  # open_clip 대신 공식 clip으로 변경
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score # 논문에 쓸 메트릭 추가

print("✅ Step 1 완료: OpenAI 공식 CLIP 및 필수 라이브러리 로드 완료.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
✅ Step 1 완료: OpenAI 공식 CLIP 및 필수 라이브러리 로드 완료.


In [3]:
import os
import zipfile
import time

zip_path = '/content/drive/MyDrive/Global_Gunshot_Dataset.zip'
extract_path = '/content/Global_Gunshot_Dataset'

if not os.path.exists(extract_path):
    print("🚀 데이터셋 압축 해제 중...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Step 2 완료: 압축 해제 성공.")
else:
    print("✅ Step 2 완료: 데이터셋이 이미 존재합니다.")

🚀 데이터셋 압축 해제 중...
✅ Step 2 완료: 압축 해제 성공.


In [5]:
import os
import pandas as pd
from collections import Counter

# 각 분할(Split) 폴더 경로 지정
splits = {
    '📁 Training Set (train)': '/content/Global_Gunshot_Dataset/train',
    '📁 Validation Set (val)': '/content/Global_Gunshot_Dataset/val',
    '📁 External Test Set (test)': '/content/Global_Gunshot_Dataset/test'
}

summary_data = []

for split_name, split_dir in splits.items():
    if os.path.exists(split_dir):
        # 우리가 만든 함수로 파일과 라벨 파싱
        _, split_labels = robust_labeling_dataset(split_dir)
        counts = Counter(split_labels)

        # 데이터 정리 (0: Entrance, 1: Exit)
        en_count = counts.get(0, 0)
        ex_count = counts.get(1, 0)
        total_count = en_count + ex_count

        summary_data.append({
            "Dataset Split": split_name,
            "Entrance Wounds (0)": f"{en_count}장",
            "Exit Wounds (1)": f"{ex_count}장",
            "Total Images": f"{total_count}장"
        })
    else:
        summary_data.append({
            "Dataset Split": split_name,
            "Entrance Wounds (0)": "경로 없음",
            "Exit Wounds (1)": "경로 없음",
            "Total Images": "경로 없음"
        })

# Pandas DataFrame으로 깔끔하게 표 출력
df_summary = pd.DataFrame(summary_data)
print("📊 [Global Gunshot Dataset 분할별 데이터 분포 요약]")
display(df_summary)

📊 [Global Gunshot Dataset 분할별 데이터 분포 요약]


,Dataset Split,Entrance Wounds (0),Exit Wounds (1),Total Images
0,📁 Training Set (train),773장,538장,1311장
1,📁 Validation Set (val),206장,122장,328장
2,📁 External Test Set (test),1883장,671장,2554장


In [6]:
import torch
import clip  # OpenAI 순정 CLIP 패키지
from torchvision import transforms

# 💡 1. OpenAI CLIP 모델을 불러오면서 전용 전처리 엔진(preprocess)을 획득합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

class GunshotDataset(Dataset):
    def __init__(self, stage_dir, transform=None):
        self.img_paths = []  # 변수명 명확화 (files 대신 img_paths)
        self.labels = []
        self.transform = transform

        for root, dirs, files_in_dir in os.walk(stage_dir):
            folder_name = os.path.basename(root).lower()

            if 'en' in folder_name:
                label = 0
            elif 'ex' in folder_name:
                label = 1
            else:
                continue

            for file in files_in_dir:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.img_paths.append(os.path.join(root, file))
                    self.labels.append(label)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# 💡 2. 수동 transform 대신 CLIP 정품 전처리 엔진(clip_preprocess)을 강제 주입합니다.
base_path = '/content/Global_Gunshot_Dataset'

train_loader = DataLoader(GunshotDataset(os.path.join(base_path, 'train'), clip_preprocess), batch_size=32, shuffle=True)
val_loader = DataLoader(GunshotDataset(os.path.join(base_path, 'val'), clip_preprocess), batch_size=32, shuffle=False)
test_loader = DataLoader(GunshotDataset(os.path.join(base_path, 'test'), clip_preprocess), batch_size=32, shuffle=False)

print(f"✅ OpenAI CLIP 최적화 로더 세팅 완결")
print(f"📊 수량 검증 -> Train: {len(train_loader.dataset)}장 | Val: {len(val_loader.dataset)}장 | Test: {len(test_loader.dataset)}장")

100%|███████████████████████████████████████| 338M/338M [00:05<00:00, 67.0MiB/s]


✅ OpenAI CLIP 최적화 로더 세팅 완결
📊 수량 검증 -> Train: 1311장 | Val: 328장 | Test: 2554장


In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
import clip
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# ================================================================= #
# ⚙️ Step 1: 환경 세팅 및 가중치 경로 지정 (preference-aligned)
# ================================================================= #
save_dir = '/content/drive/MyDrive/CCMEO_Benchmark_Results'
os.makedirs(save_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 앞선 셀에서 로드한 clip_model을 그대로 재활용하거나 안전하게 재로드합니다.
# (ViT-B/32 정품 가중치 사용)
clip_model, _ = clip.load("ViT-B/32", device=device)

# ================================================================= #
# 📝 Pipeline 1: Pure Zero-shot CLIP Evaluation (CCMEO & GuWID-UnB)
# ================================================================= #
print("\n🚀 [Pipeline 1] Pure Zero-shot CLIP 평가 시작")
clip_model.eval()

# 법의학적 타겟 텍스트 프롬프트 토큰화 (0: Entrance, 1: Exit)
text_inputs = clip.tokenize([
    "A forensic photograph of a gunshot entrance wound showing abrasion collar",
    "A forensic photograph of a gunshot exit wound showing lacerated stellar skin flaps"
]).to(device)

def evaluate_zeroshot(loader, desc_name):
    zs_probs, zs_preds, zs_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"🔮 CLIP Zero-shot [{desc_name}]"):
            imgs = imgs.to(device)
            logits_per_image, _ = clip_model(imgs, text_inputs)
            probs = logits_per_image.softmax(dim=-1).cpu().numpy()

            zs_probs.extend(probs[:, 1]) # Exit wound(1)에 대한 확률값
            zs_preds.extend(np.argmax(probs, axis=1))
            zs_labels.extend(labels.numpy())

    print(f"  📊 {desc_name} 결과:")
    print(f"    - AUC: {roc_auc_score(zs_labels, zs_probs):.4f}")
    print(f"    - ACC: {accuracy_score(zs_labels, zs_preds):.4f}")
    print(f"    - F1 : {f1_score(zs_labels, zs_preds, zero_division=0):.4f}")
    return roc_auc_score(zs_labels, zs_probs)

# 내부 검증 및 외부 검증 수치 동시 도출
zs_val_auc = evaluate_zeroshot(val_loader, "Internal Val (CCMEO)")
zs_test_auc = evaluate_zeroshot(test_loader, "External Test (GuWID)")


# ================================================================= #
# 🏗️ Pipeline 2: Linear Probe CLIP Framework
# ================================================================= #
print("\n🚀 [Pipeline 2] Linear Probe CLIP 설계 및 학습 가동")

class LinearProbeCLIP(nn.Module):
    def __init__(self, baseline_clip):
        super().__init__()
        self.visual_encoder = baseline_clip.visual
        # CLIP의 비주얼 백본 가중치는 철저하게 고정 (Linear Probe 규격)
        for param in self.visual_encoder.parameters():
            param.requires_grad = False

        # ViT-B/32의 임베딩 차원인 512차원을 받아 이진 분류하는 선형 레이어 설계
        self.classifier = nn.Linear(512, 2)

    def forward(self, x):
        with torch.no_grad():
            # CLIP 모델의 정밀도 가중치 타입 변환 호환
            feat = self.visual_encoder(x.type(clip_model.dtype))
        return self.classifier(feat.float())

# 모델 선언 및 최적화기 세팅
probe_model = LinearProbeCLIP(clip_model).to(device)
optimizer = optim.AdamW(probe_model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

# 1.48x 클래스 imbalance 페널티 보정 반영
class_weights = torch.tensor([1.0, 1.48], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_lp_path = os.path.join(save_dir, "linear_probe_clip_best.pth")
best_val_auc = 0.0

# 학술 벤치마크 안정성을 위한 2에폭 최적화 루프
for epoch in range(1, 3):
    probe_model.train()
    total_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"🏋️ Linear Probe [Train Epoch {epoch}/2]"):
        optimizer.zero_grad()
        outputs = probe_model(imgs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Epoch 종료 후 Internal Validation 성능 측정
    probe_model.eval()
    v_probs, v_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            outputs = probe_model(imgs.to(device))
            v_probs.extend(torch.softmax(outputs, dim=-1)[:, 1].cpu().numpy())
            v_labels.extend(labels.numpy())

    val_auc = roc_auc_score(v_labels, v_probs)
    print(f"  📉 Loss: {total_loss/len(train_loader):.4f} | Val ROC-AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(probe_model.state_dict(), best_lp_path)
        print(f"  🌟 [최고점 저장] 내부 검증 스코어 갱신 완료!")

# ================================================================= #
# 🏁 최종 채점: Linear Probe 최종 외부 검증 (External Test)
# ================================================================= #
print("\n📝 Linear Probe CLIP 최종 성적표 산출 중...")
probe_model.load_state_dict(torch.load(best_lp_path))
probe_model.eval()

t_probs, t_preds, t_labels = [], [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="📝 GuWID 수능 채점 중"):
        logits = probe_model(imgs.to(device))
        t_probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
        t_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        t_labels.extend(labels.numpy())

print("\n✅ Linear Probe CLIP 외부 검증 성적표:")
print(f"  - AUC: {roc_auc_score(t_labels, t_probs):.4f}")
print(f"  - ACC: {accuracy_score(t_labels, t_preds):.4f}")
print(f"  - PRE: {precision_score(t_labels, t_preds, zero_division=0):.4f}")
print(f"  - REC: {recall_score(t_labels, t_preds, zero_division=0):.4f}")
print(f"  - F1 : {f1_score(t_labels, t_preds, zero_division=0):.4f}")


🚀 [Pipeline 1] Pure Zero-shot CLIP 평가 시작


🔮 CLIP Zero-shot [Internal Val (CCMEO)]: 100%|██████████| 11/11 [00:14<00:00,  1.31s/it]


  📊 Internal Val (CCMEO) 결과:
    - AUC: 0.4102
    - ACC: 0.4116
    - F1 : 0.4533


🔮 CLIP Zero-shot [External Test (GuWID)]: 100%|██████████| 80/80 [00:58<00:00,  1.36it/s]


  📊 External Test (GuWID) 결과:
    - AUC: 0.3577
    - ACC: 0.3156
    - F1 : 0.3359

🚀 [Pipeline 2] Linear Probe CLIP 설계 및 학습 가동


...

  📉 Loss: 0.6401 | Val ROC-AUC: 0.7643
  🌟 [최고점 저장] 내부 검증 스코어 갱신 완료!


🏋️ Linear Probe [Train Epoch 2/2]: 100%|██████████| 41/41 [00:36<00:00,  1.13it/s]


  📉 Loss: 0.5794 | Val ROC-AUC: 0.7855
  🌟 [최고점 저장] 내부 검증 스코어 갱신 완료!

📝 Linear Probe CLIP 최종 성적표 산출 중...


📝 GuWID 수능 채점 중: 100%|██████████| 80/80 [00:56<00:00,  1.40it/s]


✅ Linear Probe CLIP 외부 검증 성적표:
  - AUC: 0.8280
  - ACC: 0.7357
  - PRE: 0.4981
  - REC: 0.7973
  - F1 : 0.6132
